# Batch Instance Segmentation using SAM2 Automatic Mask Generator

This notebook performs batch instance segmentation on all images in an input directory using SAM2's automatic mask generator.

In [1]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

## 1. Configuration

In [2]:
INPUT_DIR  = "/workspace/sam/input"
OUTPUT_DIR = "/workspace/sam/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# SAM2 checkpoint and config (adjust paths as needed)
CHECKPOINT = "/workspace/segment-anything-2/checkpoints/sam2.1_hiera_large.pt"
CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"   # relative to segment-anything-2 root

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. Load SAM2 Model and Mask Generator

In [3]:
# Build the model
sam2 = build_sam2(CONFIG, CHECKPOINT, device=device)

# Automatic mask generator with typical parameters
mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2,
    points_per_side=32,          # number of points per side for point sampling
    pred_iou_thresh=0.88,        # filter masks by predicted IoU
    stability_score_thresh=0.95, # filter masks by stability score
    crop_n_layers=1,             # number of layers to crop and refine
    crop_n_points_downscale_factor=2,
    min_mask_region_area=100,    # remove small disconnected regions
    output_mode="binary_mask",   # returns binary masks (bool)
)

print("SAM2 mask generator ready.")

SAM2 mask generator ready.


## 3. Process Each Image

In [4]:
# Get list of image files (supports common formats)
image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
image_files = [f for f in os.listdir(INPUT_DIR) 
               if f.lower().endswith(image_extensions)]

if not image_files:
    print(f"No images found in {INPUT_DIR}")
else:
    print(f"Found {len(image_files)} images.")

for img_file in image_files:
    img_path = os.path.join(INPUT_DIR, img_file)
    print(f"\nProcessing: {img_file}")

    # Load image
    image = np.array(Image.open(img_path).convert("RGB"))

    # Generate masks
    masks = mask_generator.generate(image)

    if not masks:
        print("  No masks generated.")
        continue

    print(f"  Generated {len(masks)} masks.")

    # Create a subfolder for this image's outputs
    base_name = os.path.splitext(img_file)[0]
    out_subdir = os.path.join(OUTPUT_DIR, base_name)
    os.makedirs(out_subdir, exist_ok=True)

    # Also save a composite visualization (original image with all masks overlaid)
    plt.figure(figsize=(12, 10))
    plt.imshow(image)
    for mask_data in masks:
        mask = mask_data["segmentation"]  # binary mask (H, W)
        # Show each mask with random color (semi-transparent)
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
        h, w = mask.shape[-2:]
        mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
        plt.imshow(mask_image)
    plt.axis("off")
    plt.title(f"{base_name} - all masks")
    plt.savefig(os.path.join(out_subdir, "all_masks_overlay.png"), bbox_inches="tight", pad_inches=0)
    plt.close()
    print(f"  Saved overlay to {out_subdir}/all_masks_overlay.png")

    # Save each individual mask as a separate PNG (transparent background, object in white)
    # Also save cropped objects with transparent background
    for i, mask_data in enumerate(masks):
        mask = mask_data["segmentation"]  # bool array
        # Create an RGBA image: keep original pixels where mask is True, transparent elsewhere
        rgba = np.zeros((image.shape[0], image.shape[1], 4), dtype=np.uint8)
        rgba[:, :, :3] = image
        rgba[:, :, 3] = (mask * 255).astype(np.uint8)

        # Save individual mask image
        mask_img = Image.fromarray(rgba, "RGBA")
        mask_img.save(os.path.join(out_subdir, f"mask_{i:03d}.png"))

    print(f"  Saved {len(masks)} individual masks to {out_subdir}/")

print("\nAll images processed. Output saved to:", OUTPUT_DIR)

Found 1 images.

Processing: multi_box2.png
  Generated 30 masks.
  Saved overlay to /workspace/sam/output/multi_box2/all_masks_overlay.png
  Saved 30 individual masks to /workspace/sam/output/multi_box2/

All images processed. Output saved to: /workspace/sam/output
